# 01 · Synthetic training data with SMOTE (SMOTEN)

**Goal:** build a larger, class-balanced training set from 183 survey responses.

**Why SMOTEN and not plain SMOTE?** Every survey answer is a category code (`"never" = 0 … "always" = 4`,
`major = 0..4`). Plain SMOTE draws a point on the line between two neighbours and would produce answers
like `2.37` or `major = 1.6` that nobody could give. **SMOTEN** (SMOTE-*Nominal*) instead finds each
seed row's *k* nearest same-class neighbours under the Value Difference Metric and takes the most
common answer per question, so every synthetic answer is a real option.

**Leakage safeguards**
1. Split into train / test **first**; the test set stays untouched and is never resampled.
2. Oversample only the training rows.
3. In model training (`dropout_risk.models`) SMOTEN sits *inside* the pipeline, so during CV it only sees
   the training folds. This notebook writes the same training-set augmentation to disk for inspection.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from dropout_risk import data as D
from dropout_risk.config import load_config
from dropout_risk.evaluation import BLUE, ORANGE
from dropout_risk.synthetic import build_synthetic_dataset, write_processed_datasets

pd.set_option("display.max_columns", 50)
cfg = load_config()
cfg["smote"], cfg["split"]

({'k_neighbors': 5, 'samples_per_class': 400}, {'test_size': 0.25})

## 1 · Load and encode the survey
Arabic answers are mapped to integer codes in `dropout_risk/data.py`; unknown answers raise an error.

In [2]:
raw = D.load_raw(cfg["data"]["raw_path"], cfg["data"]["sheet"])
X, y = D.encode_survey(raw)
print(X.shape, "| label balance:", y.value_counts().to_dict())
X.head()

(183, 32) | label balance: {0: 133, 1: 50}


,is_male,major,degree_level,study_year,residence_urbanity,is_private_university,ever_used_ai,ai_study_frequency,ai_replaces_research,analytical_skills_effect,independent_learning,motivation_effect,over_reliance,ai_cheating_view,submitted_full_project,reviews_content,content_errors,used_ai_in_exam,saves_time,flagged_by_professor,professor_was_right,detector_knowledge,easy_to_detect,ai_improved_education,ai_replaces_search_engines,uses_chatgpt,uses_gemini,uses_claude,uses_copilot,uses_other_tool,gave_advice,learned_prompting
0,0,0,1,5,1,0,1,3,4,0,2,1,2,1,0,1,2,0,0,0,0,0,1,0,4,1,1,0,0,0,1,0
1,0,0,1,3,3,0,1,3,4,1,2,2,2,1,0,2,2,0,2,0,0,3,2,2,2,1,1,0,0,0,0,2
2,1,0,1,4,2,0,1,4,3,2,2,1,0,1,0,2,2,0,2,0,0,2,1,2,2,1,0,0,0,0,0,2
3,1,1,1,4,3,0,1,4,3,2,2,2,1,1,1,1,2,0,2,1,3,3,2,2,2,0,1,1,0,0,1,0
4,0,2,1,4,3,0,1,1,0,1,1,1,0,2,0,2,2,0,1,0,0,1,2,0,1,0,1,0,0,0,1,0


## 2 · Split first, then oversample the training rows only

In [3]:
X_train, X_test, y_train, y_test = D.split(X, y, cfg)
syn = build_synthetic_dataset(X_train, y_train, cfg)
pd.crosstab(syn["is_synthetic"].map({0: "real", 1: "synthetic"}), syn["drop_label"], margins=True)

drop_label,0,1,All
is_synthetic,,,
real,100,37,137
synthetic,300,363,663
All,400,400,800


## 3 · Do synthetic rows look like real ones?
Share of each answer per question, real training rows vs synthetic rows.

In [4]:
real, fake = syn[syn.is_synthetic == 0], syn[syn.is_synthetic == 1]
cols = ["ai_study_frequency", "over_reliance", "motivation_effect", "analytical_skills_effect",
        "reviews_content", "submitted_full_project", "detector_knowledge", "study_year"]
fig, axes = plt.subplots(2, 4, figsize=(14, 5.5))
for ax, col in zip(axes.flat, cols):
    share = pd.DataFrame({"real": real[col].value_counts(normalize=True),
                          "synthetic": fake[col].value_counts(normalize=True)}).fillna(0).sort_index()
    share.plot.bar(ax=ax, color=[BLUE, ORANGE], width=0.75, legend=False, rot=0)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("answer code")
axes.flat[0].legend(frameon=False)
fig.suptitle("Answer distributions: real vs SMOTEN rows (training set)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()

Differences are expected: the synthetic rows are ~73% drop-class (the minority class needed more
oversampling), so drop-associated answers become more frequent overall.

In [5]:
# Compare within each class, which is the fair comparison
summary = []
for label in (0, 1):
    r, f = real[real.drop_label == label], fake[fake.drop_label == label]
    for col in X.columns:
        summary.append({"class": label, "feature": col, "real_mean": r[col].mean(), "synthetic_mean": f[col].mean()})
summary = pd.DataFrame(summary).assign(abs_diff=lambda d: (d.real_mean - d.synthetic_mean).abs())
summary.sort_values("abs_diff", ascending=False).head(10).round(3)

,class,feature,real_mean,synthetic_mean,abs_diff
33,1,major,0.703,0.000,0.703
1,0,major,0.730,0.087,0.643
53,1,detector_knowledge,1.486,1.110,0.376
48,1,content_errors,2.297,2.634,0.336
35,1,study_year,3.730,4.055,0.325
36,1,residence_urbanity,2.595,2.904,0.309
21,0,detector_knowledge,1.450,1.143,0.307
59,1,uses_claude,0.297,0.000,0.297
4,0,residence_urbanity,2.580,2.877,0.297
41,1,analytical_skills_effect,0.703,0.413,0.289


## 4 · Diversity check: a known SMOTEN weakness
Taking the per-question **mode** of 5 neighbours erases rare answers (e.g. every synthetic drop-class
row ends up in the most common major) and produces many identical rows. Oversampling to 400 per class
therefore adds *copies of typical answer patterns*, not new information. The test set is never resampled, but it caps how much SMOTEN can help. Lowering
`smote.samples_per_class` in `config/config.yaml` (e.g. to 100 = just balancing) reduces duplication.

In [6]:
unique = fake.drop_duplicates()
print(f"synthetic rows: {len(fake)}, distinct: {len(unique)}")
print("distinct synthetic rows per class:", unique["drop_label"].value_counts().to_dict())
shrunk = [c for c in X.columns if fake[c].nunique() < real[c].nunique()]
print(f"{len(shrunk)} / {X.shape[1]} questions lose answer options in the synthetic rows:", shrunk)

synthetic rows: 663, distinct: 108
distinct synthetic rows per class: {0: 76, 1: 32}
10 / 32 questions lose answer options in the synthetic rows: ['major', 'degree_level', 'residence_urbanity', 'is_private_university', 'ever_used_ai', 'ai_study_frequency', 'ai_replaces_research', 'content_errors', 'uses_copilot', 'uses_other_tool']


## 5 · Save the processed datasets

In [7]:
paths = write_processed_datasets(cfg)
{k: str(v.relative_to(cfg["data"]["processed_dir"].parents[1])) for k, v in paths.items()}

{'encoded_full': 'data\\processed\\survey_encoded.csv',
 'train_real': 'data\\processed\\train_real.csv',
 'test_real': 'data\\processed\\test_real.csv',
 'train_synthetic': 'data\\processed\\train_smoten.csv'}

**Next:** `uv run dropout-risk` runs feature engineering, feature selection, and tunes / evaluates the five
models. Results go to `outputs/`.